# Customer Segmentation System
## End-to-End ML Solution for Marketing Intelligence

**Author**: Senior ML Engineer
**Objective**: Build a production-ready customer segmentation system using K-Means clustering

### Business Context
This notebook implements a complete customer segmentation pipeline that:
- Identifies distinct customer groups based on behavior and demographics
- Provides actionable insights for targeted marketing campaigns
- Serves as the ML core for a full-stack deployment system

### Workflow
1. Data Loading & Validation
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Scaling
4. Optimal Cluster Selection (Elbow Method)
5. K-Means Clustering
6. Cluster Profiling & Business Insights
7. Model Persistence & Deployment Prep

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from src.data_processing import DataProcessor
from src.modeling import CustomerSegmentation
from src.utils import setup_plot_style, save_cluster_report, plot_cluster_distribution, get_segment_names

setup_plot_style()
%matplotlib inline
print('✓ Environment ready')

## 1. Data Loading & Validation

We'll create sample Mall Customers data with:
- CustomerID, Gender, Age, Annual Income, Spending Score

In [ ]:
# Create sample dataset
np.random.seed(42)
n_samples = 200

data = {
    'CustomerID': range(1, n_samples + 1),
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'Age': np.random.randint(18, 70, n_samples),
    'Annual Income (k$)': np.random.randint(15, 140, n_samples),
    'Spending Score (1-100)': np.random.randint(1, 100, n_samples)
}

df_raw = pd.DataFrame(data)
df_raw.to_csv('../data/raw/mall_customers.csv', index=False)

print(f'Dataset shape: {df_raw.shape}')
print(f'\nFirst 5 rows:')
df_raw.head()

In [ ]:
# Data validation
processor = DataProcessor()
df_clean = processor.clean_data(df_raw)

print('Data Summary:')
print(f'Total records: {len(df_clean)}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
print(f'Duplicates: {df_clean.duplicated().sum()}')
print(f'\nData types:\n{df_clean.dtypes}')

## 2. Exploratory Data Analysis (EDA)

Visualize distributions and relationships

In [ ]:
# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df_clean['Age'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].hist(df_clean['Annual Income (k$)'], bins=20, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Annual Income Distribution')
axes[0, 1].set_xlabel('Income (k$)')
axes[0, 1].set_ylabel('Frequency')

axes[1, 0].hist(df_clean['Spending Score (1-100)'], bins=20, color='salmon', edgecolor='black')
axes[1, 0].set_title('Spending Score Distribution')
axes[1, 0].set_xlabel('Spending Score')
axes[1, 0].set_ylabel('Frequency')

gender_counts = df_clean['Gender'].value_counts()
axes[1, 1].bar(gender_counts.index, gender_counts.values, color=['steelblue', 'coral'])
axes[1, 1].set_title('Gender Distribution')
axes[1, 1].set_xlabel('Gender')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/outputs/eda_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_clean['Annual Income (k$)'], df_clean['Spending Score (1-100)'], alpha=0.6, c='purple')
axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Spending Score')
axes[0].set_title('Income vs Spending Score')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df_clean['Age'], df_clean['Spending Score (1-100)'], alpha=0.6, c='orange')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Spending Score')
axes[1].set_title('Age vs Spending Score')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/eda_relationships.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Feature Engineering & Scaling

Prepare features for clustering

In [ ]:
# Select features for clustering
feature_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

df_processed, X_scaled = processor.prepare_features(df_clean, feature_cols)

print(f'Features selected: {feature_cols}')
print(f'Scaled data shape: {X_scaled.shape}')
print(f'\nScaled data sample (first 5 rows):\n{X_scaled[:5]}')
print(f'\nFeature means after scaling: {X_scaled.mean(axis=0)}')
print(f'Feature stds after scaling: {X_scaled.std(axis=0)}')

## 4. Optimal Cluster Selection - Elbow Method

Determine the best number of clusters

In [ ]:
# Find optimal k
segmentation_model = CustomerSegmentation(random_state=42)
optimal_k, inertias = segmentation_model.find_optimal_clusters(X_scaled, k_range=range(2, 11))

print(f'Optimal number of clusters: {optimal_k}')
print(f'\nInertia values for k=2 to k=10:')
for k, inertia in zip(range(2, 11), inertias):
    print(f'  k={k}: {inertia:.2f}')

In [ ]:
# Plot elbow curve
fig = segmentation_model.plot_elbow(k_range=range(2, 11))
plt.savefig('../data/outputs/elbow_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. K-Means Clustering

Apply clustering with optimal k

In [ ]:
# Fit model
segmentation_model.fit(X_scaled)

# Get predictions
df_processed['Cluster'] = segmentation_model.predict(X_scaled)

# Model metrics
metrics = segmentation_model.get_metrics()
print('Model Performance:')
for key, value in metrics.items():
    print(f'  {key}: {value}')

print(f'\nCluster distribution:')
print(df_processed['Cluster'].value_counts().sort_index())

In [ ]:
# Visualize clusters (2D projections)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for cluster in sorted(df_processed['Cluster'].unique()):
    cluster_data = df_processed[df_processed['Cluster'] == cluster]
    axes[0].scatter(cluster_data['Annual Income (k$)'], cluster_data['Spending Score (1-100)'], 
                   label=f'Cluster {cluster}', alpha=0.6, s=50)

axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Spending Score (1-100)')
axes[0].set_title('Customer Segments: Income vs Spending')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for cluster in sorted(df_processed['Cluster'].unique()):
    cluster_data = df_processed[df_processed['Cluster'] == cluster]
    axes[1].scatter(cluster_data['Age'], cluster_data['Spending Score (1-100)'], 
                   label=f'Cluster {cluster}', alpha=0.6, s=50)

axes[1].set_xlabel('Age')
axes[1].set_ylabel('Spending Score (1-100)')
axes[1].set_title('Customer Segments: Age vs Spending')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/cluster_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Cluster Profiling & Business Insights

Analyze each segment and derive actionable strategies

In [ ]:
# Cluster statistics
print('Cluster Profiles:\n')
for cluster_id in sorted(df_processed['Cluster'].unique()):
    cluster_data = df_processed[df_processed['Cluster'] == cluster_id]
    print(f'\n=== CLUSTER {cluster_id} ===')
    print(f'Size: {len(cluster_data)} customers ({len(cluster_data)/len(df_processed)*100:.1f}%)')
    print(f'\nAverage Profile:')
    print(f'  Age: {cluster_data["Age"].mean():.1f} years')
    print(f'  Income: ${cluster_data["Annual Income (k$)"].mean():.1f}k')
    print(f'  Spending Score: {cluster_data["Spending Score (1-100)"].mean():.1f}/100')
    print(f'  Gender: {cluster_data["Gender"].value_counts().to_dict()}')

In [ ]:
# Assign business names and strategies
segment_info = get_segment_names()

print('\n' + '='*80)
print('BUSINESS SEGMENT ANALYSIS & MARKETING STRATEGIES')
print('='*80 + '\n')

for cluster_id in sorted(df_processed['Cluster'].unique()):
    if cluster_id in segment_info:
        info = segment_info[cluster_id]
        cluster_data = df_processed[df_processed['Cluster'] == cluster_id]
        
        print(f"\nSegment {cluster_id}: {info['name']}")
        print(f"Description: {info['description']}")
        print(f"Size: {len(cluster_data)} customers")
        print(f"Strategy: {info['strategy']}")
        print('-' * 80)

In [ ]:
# Save cluster report
report = save_cluster_report(df_processed, 'Cluster', '../data/outputs/cluster_report.json')
print('✓ Cluster report saved to data/outputs/cluster_report.json')

In [ ]:
# Cluster distribution plot
fig = plot_cluster_distribution(df_processed, 'Cluster')
plt.savefig('../data/outputs/cluster_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Model Persistence & Deployment Preparation

Save model and processed data for production use

In [ ]:
# Save model
segmentation_model.save_model('../data/outputs/segmentation_model.pkl')
print('✓ Model saved to data/outputs/segmentation_model.pkl')

# Save processed data with clusters
df_processed.to_csv('../data/processed/customers_with_clusters.csv', index=False)
print('✓ Processed data saved to data/processed/customers_with_clusters.csv')

# Save scaler
import joblib
joblib.dump(processor.scaler, '../data/outputs/scaler.pkl')
print('✓ Scaler saved to data/outputs/scaler.pkl')

## Summary & Next Steps

### Key Findings
1. Successfully identified distinct customer segments using K-Means clustering
2. Each segment has unique characteristics requiring tailored marketing approaches
3. Model ready for production deployment via API

### Deployment Architecture
- **Backend**: FastAPI serves predictions via REST endpoints
- **Frontend**: Next.js dashboard for business users
- **Model**: Persisted using joblib for fast loading
- **Scalability**: Stateless API design enables horizontal scaling

### Business Impact
- Targeted marketing campaigns per segment
- Improved customer retention through personalization
- Optimized marketing budget allocation
- Data-driven decision making for product teams